# Stage 1 Launcher — Longformer-Large NER training with global-attention dropout

**What this does:** trains the encoder + NER head on `data/labeled/final/train.jsonl` (17,593 articles, Sonnet-relabeled) with per-sample global-attention dropout (p=0.3, warmup over first 30% of training).

**Goal:** produce an encoder + NER head that works in CLS-only global attention regime (e2e inference) without losing the entity-aware regime.

**Estimated time:** ~2.5 hours on A100 80GB at batch_size=16.

**Save pattern:** local-first (writes to `/content/stage1_local/`), syncs to Drive at the end. If Colab disconnects mid-training, intermediate checkpoints are lost — but final + best should make it to Drive.

**Pre-flight signal:** watch the **CLS-only NER F1** at the end of each epoch. If it stays near zero after Epoch 1, kill the run and pivot to a different approach (heuristic priming or retraining without entity-aware regime entirely).

In [ ]:
# 1. Mount Drive & check GPU
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set project path — UPDATE if your Drive mount layout differs
import os
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
assert os.path.exists(f"{PROJECT_PATH}/scripts/training/train_two_stage.py"), "train_two_stage.py not found!"
assert os.path.exists(f"{PROJECT_PATH}/data/labeled/final/train.jsonl"), "training data not found!"

# Local-first save: checkpoints write to /content/, sync to Drive at the end
LOCAL_CKPT_DIR = "/content/stage1_local"
DRIVE_CKPT_DIR = f"{PROJECT_PATH}/checkpoints/stage1_ner_large_v2"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"Project       : {PROJECT_PATH}")
print(f"Local ckpts   : {LOCAL_CKPT_DIR}")
print(f"Drive ckpts   : {DRIVE_CKPT_DIR}")

In [ ]:
# 3. Install deps
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 4. Run Stage 1
#    Defaults: Longformer-Large, hidden=1024, max_len=2048, CRF NER, AMP fp16
#    Dropout: p=0.3, warmup over first 30% of Stage-1 optimizer steps
#    Dual-eval: validation runs twice per epoch (entity-aware + CLS-only)
#
#    Memory: A100 80GB at batch=16 should peak ~60 GB. If first few batches
#    show <50% util, kill and rerun with --batch_size 20.
#
#    Watch the log for the per-epoch line:
#      Validation [CLS-only]: ner_f1=...   (this is the e2e signal)

!cd {PROJECT_PATH} && python scripts/training/train_two_stage.py \
    --stage 1 \
    --batch_size 24 \
    --stage1_epochs 5 \
    --learning_rate 2e-5 \
    --global_attn_dropout_prob 0.3 \
    --stage1_dropout_warmup_frac 0.3 \
    --gradient_checkpointing \
    --stage1_checkpoint_dir {LOCAL_CKPT_DIR}

In [ ]:
# 5. Show local checkpoints (these are what's safe to copy to Drive)
import os
print(f"Local checkpoints in {LOCAL_CKPT_DIR}:")
for f in sorted(os.listdir(LOCAL_CKPT_DIR)):
    path = os.path.join(LOCAL_CKPT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {f:40s} {size:8.1f} MB")

# Also show the latest training log
import glob
logs = sorted(glob.glob(f"{PROJECT_PATH}/logs/two_stage_training_*.log"))
if logs:
    print(f"\nLatest log: {logs[-1]} (last 40 lines)")
    with open(logs[-1]) as fh:
        for line in fh.readlines()[-40:]:
            print(f"  {line.rstrip()}")

In [ ]:
# 6. Sync local checkpoints -> Drive (local-first pattern)
#    This is the step that prevents Drive FUSE from dropping large writes.
import shutil, os

src_dir = LOCAL_CKPT_DIR
dst_dir = DRIVE_CKPT_DIR
os.makedirs(dst_dir, exist_ok=True)

print(f"Syncing {src_dir} -> {dst_dir}")
for f in sorted(os.listdir(src_dir)):
    src = os.path.join(src_dir, f)
    dst = os.path.join(dst_dir, f)
    if os.path.isfile(src):
        try:
            shutil.copy2(src, dst)
            size = os.path.getsize(dst) / 1e6
            print(f"  {f:40s} -> Drive OK  ({size:.1f} MB)")
        except Exception as e:
            print(f"  {f:40s} -> FAILED: {e}")

# Flush Drive buffer to force FUSE to actually write (this was the fix
# that finally got Stage 3 checkpoints to land on Drive in April)
import time
from google.colab import drive
print('\nFlushing Drive...')
drive.flush_and_unmount()
time.sleep(3)
drive.mount('/content/drive')
time.sleep(5)  # let FUSE reconnect
print('Drive remounted.')

In [ ]:
# 7. Verify the checkpoints actually landed on Drive
import os
print(f"Drive checkpoints in {DRIVE_CKPT_DIR}:")
if os.path.exists(DRIVE_CKPT_DIR):
    for f in sorted(os.listdir(DRIVE_CKPT_DIR)):
        path = os.path.join(DRIVE_CKPT_DIR, f)
        if os.path.isfile(path):
            size = os.path.getsize(path) / 1e6
            print(f"  {f:40s} {size:8.1f} MB")
else:
    print(f"  Directory not visible yet — Drive FUSE may still be reconnecting.")
    print(f"  Local copies are at: {LOCAL_CKPT_DIR}")

In [ ]:
# 8. (Optional) Terminate runtime to stop billing
#    Only run when you've confirmed the Drive sync worked (cell 7 above shows files).
from google.colab import runtime
runtime.unassign()